# Self-Explaining Hybrid CNN-ViT Deepfake Detector
Run this notebook on Google Colab or Kaggle (GPU runtime recommended).

In [ ]:
# Mount Google Drive (Colab only) — skip on Kaggle
import os
IN_COLAB = 'google.colab' in str(__import__('sys').modules)
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/deepfake_outputs'
else:
    SAVE_DIR = '/kaggle/working/deepfake_outputs'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Save dir:', SAVE_DIR)

In [ ]:
# Install dependencies
!pip install torch torchvision Pillow numpy matplotlib -q

In [ ]:
# Clone or upload your repo, then add to path
import sys
# If running from repo root:
sys.path.insert(0, '.')
# Or if cloned: sys.path.insert(0, '/content/your-repo')

In [ ]:
import torch
from src.deepfake_detector.config import ModelConfig, PreprocessorConfig, MatrixConfig, ScorecardWeights
from src.deepfake_detector.models import HybridCNNViT
from src.deepfake_detector.preprocessor import Preprocessor
from src.deepfake_detector.evaluator import Evaluator
from src.deepfake_detector.pretty_printer import PrettyPrinter
from src.deepfake_detector.train import TrainConfig, train

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
# --- Config (Colab/Kaggle friendly small model) ---
model_cfg = ModelConfig(
    cnn_out_channels=64,
    patch_size=4,       # smaller patches for 28x28 feature map
    vit_num_layers=2,
    vit_num_heads=4,
    vit_embed_dim=128,
)
pre_cfg = PreprocessorConfig(dataset_type='ff++')  # or 'celeb-df'

In [ ]:
# --- Training ---
# Set DATASET_ROOT to your FF++ or Celeb-DF directory
DATASET_ROOT = '/content/drive/MyDrive/datasets/ff++'  # update this

train_cfg = TrainConfig(
    dataset_root=DATASET_ROOT,
    output_dir=SAVE_DIR,
    epochs=20,
    batch_size=16,
    learning_rate=1e-4,
    optimizer='adam',
    patience=5,
    device=DEVICE,
)

best_ckpt = train(train_cfg, model_cfg, pre_cfg)
print('Best checkpoint:', best_ckpt)

In [ ]:
# --- Single image inference ---
from src.deepfake_detector.infer import run_inference

TEST_IMAGE = '/content/test_face.jpg'  # update this
report = run_inference(
    checkpoint_path=best_ckpt,
    image_path=TEST_IMAGE,
    output_dir=os.path.join(SAVE_DIR, 'inference'),
    device=DEVICE,
)
print(f'Label: {report.label} | Confidence: {report.confidence:.4f}')

# Show overlay
from IPython.display import Image as IPImage
IPImage(report.attention_map_path)

In [ ]:
# --- Stress-Test Matrix + Forensic Trust Scorecard ---
model = HybridCNNViT.load_checkpoint(best_ckpt).to(DEVICE)
preprocessor = Preprocessor(pre_cfg)
test_dataset = preprocessor.load_dataset(DATASET_ROOT, split='test')

matrix_cfg = MatrixConfig(
    jpeg_qualities=[95, 75, 50, 25],
    blur_sigmas=[1.0, 2.0],
    noise_stds=[0.05, 0.1],
    include_baseline=True,
)

evaluator = Evaluator()
stress_results = evaluator.run_stress_test_matrix(model, test_dataset, matrix_cfg, device=DEVICE)
evaluator.serialize_results(stress_results, os.path.join(SAVE_DIR, 'stress_test_matrix.json'))

weights = ScorecardWeights(accuracy_weight=0.4, iou_weight=0.4, ssim_weight=0.2)
scorecard = evaluator.compute_forensic_trust_scorecard(stress_results, weights)
evaluator.serialize_results(scorecard, os.path.join(SAVE_DIR, 'forensic_trust_scorecard.json'))

print(f'Forensic Trust Score: {scorecard.composite_score:.4f}')
print(f'  Accuracy:  {scorecard.accuracy:.4f}')
print(f'  Mean IoU:  {scorecard.mean_iou}')
print(f'  Mean SSIM: {scorecard.mean_ssim}')

In [ ]:
# --- Plot Stress-Test Matrix results ---
import matplotlib.pyplot as plt

rows = stress_results.rows
labels = [f"{r.degradation_type}\n{r.severity}" for r in rows]
accuracies = [r.accuracy for r in rows]
ious = [r.mean_iou if r.mean_iou is not None else 0 for r in rows]

x = range(len(rows))
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(x, accuracies, alpha=0.6, label='Accuracy', color='steelblue')
ax1.set_ylabel('Accuracy')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, fontsize=8)
ax2 = ax1.twinx()
ax2.plot(list(x), ious, 'o-', color='tomato', label='Mean IoU')
ax2.set_ylabel('Mean IoU')
fig.legend(loc='upper right')
plt.title('Stress-Test Matrix: Accuracy & IoU under Degradation')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'stress_test_plot.png'))
plt.show()